# 📋 claude.ai 프롬프트 / claude.ai Prompt

> **수업 활용법:** 아래 프롬프트를 [claude.ai](https://claude.ai)에 붙여넣으면 유사한 노트북을 생성할 수 있습니다.

---

### 🇰🇷 한국어 프롬프트
```
Anthropic Claude API를 활용한 주간 비즈니스 보고서 자동화 에이전트
Google Colab 노트북을 한국어로 작성해주세요. 다음을 포함하세요:

1. 가상 입력 데이터 준비:
   - 이번 주 일별 매출 딕셔너리 (월~금)
   - 고객 피드백 텍스트 리스트 (5개)
   - 이슈/이벤트 로그 리스트 (3개)

2. 섹션별 분석 함수 3개 (각각 Claude API 호출):
   - analyze_sales(data): 매출 분석 → 합계, 추이, 주목할 점
   - analyze_feedback(feedbacks): 고객 감성 분석 → 긍/부정 비율, 핵심 키워드
   - analyze_issues(issues): 이슈 분석 → 심각도 분류, 액션 아이템

3. 전체 보고서 생성 함수:
   - 3개 섹션 분석 결과 + 경영진 종합 요약 생성
   - 마크다운 포맷으로 최종 보고서 출력

4. 토큰 비용 추정 기능:
   - 각 API 호출의 토큰 수 추적
   - 총 비용 계산 (sonnet 기준)

5. 커스터마이즈 가이드:
   - 다른 업무 도메인에 적용하는 방법 설명
   - 커스터마이즈 가능한 변수 목록

한국어 주석, 초보자 친화적으로 작성.
```

### 🇺🇸 English Prompt
```
Create a Korean Google Colab notebook for a Weekly Business Report Automation Agent
using the Anthropic Claude API. Include:

1. Sample input data: daily sales dict, customer feedback list, issue log list
2. Three section analyzer functions (each calling Claude API):
   - analyze_sales(): sales trend and highlights
   - analyze_feedback(): sentiment ratio and key themes
   - analyze_issues(): severity classification and action items
3. Report generator: combine sections + executive summary → markdown output
4. Token cost estimator: track total tokens and estimated cost
5. Customization guide at the end: how to adapt to other business domains

Korean comments. Beginner-friendly. Realistic and immediately useful.
```

# FM2 실습 3: 주간 비즈니스 보고서 자동화 에이전트

## 학습 목표
- 여러 Claude API 호출을 조합하여 완성된 업무 자동화 시스템을 구현한다
- 실제 비즈니스 보고서를 자동 생성하는 파이프라인을 완성한다
- API 비용을 추정하고 관리하는 방법을 배운다

## 전체 파이프라인
```
입력 데이터 (매출/피드백/이슈)
    ↓
섹션별 분석 (Claude API × 3)
    ↓
경영진 종합 요약 생성 (Claude API × 1)
    ↓
최종 주간 보고서 (마크다운)
```

In [ ]:
!pip install anthropic -q
import getpass, anthropic

api_key = getpass.getpass("🔑 Anthropic API 키: ")
client = anthropic.Anthropic(api_key=api_key)

# 토큰 비용 추적 변수
total_input_tokens = 0
total_output_tokens = 0

print("✅ 준비 완료!")

## 1단계: 이번 주 업무 데이터 입력

실제 업무에서는 이 데이터를 ERP, CRM, 엑셀 등에서 불러오면 됩니다.

In [ ]:
# ─── 이번 주 데이터 ───────────────────────────────────
# ✏️ 이 데이터를 실제 업무 데이터로 교체하세요!

# 1. 일별 매출 데이터 (단위: 만원)
weekly_sales = {
    "월요일": 1250,
    "화요일": 980,
    "수요일": 1580,
    "목요일": 2100,
    "금요일": 1890,
}

# 2. 고객 피드백 (CRM, 리뷰 등에서 수집)
customer_feedbacks = [
    "배송이 예상보다 하루 늦게 왔지만 제품 품질은 매우 만족스럽습니다.",
    "고객센터 연결이 너무 오래 걸렸어요. 30분이나 기다렸습니다.",
    "신제품 디자인이 정말 세련됐네요! 친구들에게도 추천했어요.",
    "앱이 자꾸 튕겨요. iOS 최신 버전에서 문제가 있는 것 같아요.",
    "가격 대비 성능이 뛰어납니다. 재구매 의사 있어요.",
]

# 3. 이슈 및 이벤트 로그
issue_log = [
    "[수요일 오전 10시] 결제 서버 30분간 장애 발생 → 긴급 복구 완료. 영향 고객 약 120명",
    "[목요일] 경쟁사 B社, 유사 제품 10% 가격 인하 발표",
    "[금요일] 신규 파트너사 계약 체결 완료 (예상 월 추가 매출 500만원)",
]

total = sum(weekly_sales.values())
print(f"📊 이번 주 총 매출: {total:,}만원")
print(f"💬 고객 피드백: {len(customer_feedbacks)}건")
print(f"🚨 이슈/이벤트: {len(issue_log)}건")

## 2단계: 섹션별 분석 함수

In [ ]:
def call_claude(prompt, max_tokens=600):
    """Claude API를 호출하고 토큰 비용을 추적합니다."""
    global total_input_tokens, total_output_tokens
    
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    
    # 토큰 누적
    total_input_tokens += response.usage.input_tokens
    total_output_tokens += response.usage.output_tokens
    
    return response.content[0].text


def analyze_sales(sales_data):
    """매출 데이터를 분석합니다."""
    data_str = "\n".join([f"  {day}: {amount:,}만원" for day, amount in sales_data.items()])
    total = sum(sales_data.values())
    avg = total / len(sales_data)
    
    prompt = f"""다음 이번 주 일별 매출 데이터를 분석해주세요.

{data_str}
주간 합계: {total:,}만원 / 일평균: {avg:,.0f}만원

다음 형식으로 간결하게 분석해주세요 (3-4줄):
- 매출 추이 특징
- 주목할 날짜/수치
- 간단한 평가 (전반적으로 좋음/나쁨/보통 + 이유)"""
    
    return call_claude(prompt, max_tokens=300)


def analyze_feedback(feedbacks):
    """고객 피드백의 감성을 분석합니다."""
    feedback_str = "\n".join([f"  {i+1}. {fb}" for i, fb in enumerate(feedbacks)])
    
    prompt = f"""다음 고객 피드백들을 분석해주세요:

{feedback_str}

다음 형식으로 분석해주세요:
- 긍정/부정/중립 비율
- 주요 긍정 포인트 (키워드)
- 주요 개선 필요 포인트 (키워드)
- 즉각 대응이 필요한 항목"""
    
    return call_claude(prompt, max_tokens=300)


def analyze_issues(issues):
    """이슈 로그를 분석하고 액션 아이템을 도출합니다."""
    issues_str = "\n".join([f"  - {issue}" for issue in issues])
    
    prompt = f"""다음 이번 주 이슈/이벤트 로그를 분석해주세요:

{issues_str}

다음 형식으로 분석해주세요:
- 각 이슈 심각도 (높음/중간/낮음)
- 비즈니스 영향도 요약
- 다음 주 액션 아이템 (담당 부서 포함)"""
    
    return call_claude(prompt, max_tokens=400)

print("✅ 분석 함수 준비 완료!")

In [ ]:
# 각 섹션 분석 실행
print("📊 매출 분석 중...")
sales_analysis = analyze_sales(weekly_sales)

print("💬 고객 피드백 분석 중...")
feedback_analysis = analyze_feedback(customer_feedbacks)

print("🚨 이슈 분석 중...")
issue_analysis = analyze_issues(issue_log)

print("✅ 섹션별 분석 완료!")

## 3단계: 최종 보고서 생성

In [ ]:
from datetime import datetime

def generate_report(sales_analysis, feedback_analysis, issue_analysis, sales_data):
    """전체 분석을 바탕으로 주간 보고서를 생성합니다."""
    
    # 경영진 종합 요약 생성 (Claude가 전체를 통합)
    summary_prompt = f"""다음 이번 주 비즈니스 분석 결과를 바탕으로
경영진을 위한 3줄 종합 요약을 작성해주세요.
가장 중요한 성과, 주요 리스크, 다음 주 최우선 과제를 포함하세요.

매출 분석: {sales_analysis}
고객 감성: {feedback_analysis}
이슈 현황: {issue_analysis}"""
    
    executive_summary = call_claude(summary_prompt, max_tokens=200)
    
    # 보고서 조립
    total_sales = sum(sales_data.values())
    report_date = datetime.now().strftime("%Y년 %m월 %d일")
    
    report = f"""# 📊 주간 비즈니스 보고서
**보고 기간:** {report_date} 주간  
**생성:** Claude AI 자동 생성

---

## ⭐ 경영진 요약
{executive_summary}

---

## 💰 매출 현황
- **주간 총 매출:** {total_sales:,}만원
- **일평균 매출:** {total_sales//len(sales_data):,}만원

{sales_analysis}

---

## 👥 고객 피드백 분석
{feedback_analysis}

---

## 🚨 이슈 및 액션 아이템
{issue_analysis}

---
*본 보고서는 Claude AI가 자동 생성했습니다. 중요한 결정 전에 데이터를 재확인하세요.*
"""
    return report


# 보고서 생성
print("📝 최종 보고서 생성 중...")
final_report = generate_report(
    sales_analysis, feedback_analysis, issue_analysis, weekly_sales
)

print("\n" + "=" * 60)
print(final_report)
print("=" * 60)

In [ ]:
# 토큰 비용 추정
# claude-sonnet-4-6 기준 가격 (2024년 기준)
INPUT_PRICE_PER_TOKEN = 0.000003   # $0.003 per 1K tokens
OUTPUT_PRICE_PER_TOKEN = 0.000015  # $0.015 per 1K tokens

total_cost_usd = (
    total_input_tokens * INPUT_PRICE_PER_TOKEN +
    total_output_tokens * OUTPUT_PRICE_PER_TOKEN
)
total_cost_krw = total_cost_usd * 1350  # 환율 적용

print("💸 API 사용 비용 추정")
print("-" * 40)
print(f"  입력 토큰:  {total_input_tokens:,}개")
print(f"  출력 토큰:  {total_output_tokens:,}개")
print(f"  예상 비용:  ${total_cost_usd:.4f} (약 {total_cost_krw:.1f}원)")
print()
print(f"💡 주 5회 생성 시 월 비용 추정: 약 {total_cost_krw*20:.0f}원")

## ✏️ 커스터마이즈 가이드

이 노트북을 **본인의 업무 도메인에 맞게** 수정하는 방법:

In [ ]:
# ─── 커스터마이즈 템플릿 ─────────────────────────────────
# 아래 변수들을 본인 업무에 맞게 수정하세요!

# ✏️ 1. 입력 데이터 교체
# weekly_sales → 본인 업무 지표 (콜 수, 처리 건수, 방문자 수 등)
# customer_feedbacks → 고객 리뷰, 민원 텍스트, 회의 피드백 등
# issue_log → 장애 로그, 리스크 사항, 주요 이벤트 등

# ✏️ 2. 분석 프롬프트 수정
# analyze_sales() 내부 prompt → 본인 지표에 맞는 분석 기준
# analyze_feedback() → 분류 기준 변경 (긍/부정 외 다른 카테고리)

# ✏️ 3. 보고서 형식 수정
# generate_report() 내부 report 문자열 → 회사 보고서 양식에 맞게

# 사용 도메인 예시:
DOMAIN_EXAMPLES = {
    "인사팀":     "채용 현황, 직원 만족도 설문, 이직률, 교육 이수율",
    "고객서비스": "콜 수신량, 평균 처리 시간, CSAT 점수, 반복 민원",
    "마케팅팀":   "캠페인 클릭률, 전환율, SNS 반응, 광고 ROAS",
    "IT운영팀":   "시스템 가용률, 인시던트 수, 응답 시간, 배포 건수",
}

print("📋 적용 가능한 업무 도메인 예시:")
for domain, metrics in DOMAIN_EXAMPLES.items():
    print(f"  • {domain}: {metrics}")

## 📝 최종 정리

### 강의 2 전체 실습 요약

| 실습 | 핵심 기술 | 비즈니스 적용 |
|------|-----------|---------------|
| FM2-01 챗봇 | 시스템 프롬프트, 멀티턴 대화 | 고객 서비스 자동화 |
| FM2-02 에이전트 | Tool Use, 에이전트 루프 | 데이터 조회·계산 자동화 |
| FM2-03 보고서 | 멀티 API 파이프라인 | 주간 보고서 자동 생성 |

### AX 구현 핵심 원칙
1. **시작은 작게**: 가장 반복적인 업무 1개부터
2. **검증 먼저**: claude.ai로 프로토타입 → Colab으로 테스트
3. **비용 관리**: 토큰 추적, max_tokens 제한
4. **보안 준수**: API 키 환경변수, 개인정보 외부 전송 금지

**과제:** 본인 업무에 맞는 AI 에이전트를 직접 설계하고 구현해보세요! 🚀